In [ ]:
import subprocess, sys
from pathlib import Path
root = Path("/kaggle/temp/speakly")
root.mkdir(parents=True, exist_ok=True)
files = {'fillers.py': '"""Optional research classifier and fixed-window inference."""\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nLABELS = (\'other\', \'um\', \'uh\')\nBASE_MODEL = \'facebook/wav2vec2-base\'\nBASE_REVISION = \'0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8\'\nENCODERS = {\'wav2vec2\': (BASE_MODEL, BASE_REVISION),\n            \'wavlm\': (\'microsoft/wavlm-base-plus\', \'4c66d4806a428f2e922ccfa1a962776e232d487b\')}\nRATE = 16000\n\n\nclass FillerClassifier(nn.Module):\n    def __init__(self, encoder, unfreeze_last=0, encoder_name=\'wav2vec2\'):\n        super().__init__()\n        if encoder_name not in ENCODERS:\n            raise ValueError(\'unsupported encoder\')\n        self.encoder_name = encoder_name\n        self.encoder = encoder.eval().requires_grad_(False)\n        layers = encoder.encoder.layers\n        if not 0 <= unfreeze_last <= len(layers):\n            raise ValueError(\'invalid number of encoder layers to unfreeze\')\n        self.unfreeze_last = unfreeze_last\n        if unfreeze_last:\n            for layer in layers[-unfreeze_last:]:\n                layer.requires_grad_(True)\n        self.head = nn.Linear(encoder.config.hidden_size, len(LABELS))\n\n    def forward(self, waveform):\n        normalized = (waveform - waveform.mean(dim=-1, keepdim=True)) / (\n            waveform.var(dim=-1, keepdim=True, unbiased=False) + 1e-7\n        ).sqrt()\n        if not self.unfreeze_last:\n            self.encoder.eval()\n        with torch.set_grad_enabled(self.training and bool(self.unfreeze_last)):\n            hidden = self.encoder(normalized).last_hidden_state.mean(dim=1)\n        return self.head(hidden)\n\n\ndef save_checkpoint(model, path):\n    state = {\'head\': model.head.state_dict(), \'unfreeze_last\': model.unfreeze_last,\n             \'encoder_name\': model.encoder_name}\n    if model.unfreeze_last:\n        state[\'encoder\'] = model.encoder.state_dict()\n    torch.save(state, path)\n\n\ndef load_checkpoint(model, path):\n    state = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if state.get(\'kind\') == \'frame_filler\' and not getattr(model, \'frame_output\', False):\n        raise ValueError(\'frame checkpoint requires the frame classifier\')\n    if \'head\' not in state:\n        model.head.load_state_dict(state)\n        return\n    if state.get(\'encoder_name\', \'wav2vec2\') != model.encoder_name:\n        raise ValueError(\'checkpoint encoder does not match model\')\n    model.head.load_state_dict(state[\'head\'])\n    if \'encoder\' in state:\n        model.encoder.load_state_dict(state[\'encoder\'])\n\n\ndef checkpoint_encoder(path):\n    state = torch.load(path, map_location=\'cpu\', weights_only=True)\n    name = state.get(\'encoder_name\', \'wav2vec2\')\n    if name not in ENCODERS:\n        raise ValueError(\'unsupported checkpoint encoder\')\n    return name\n\n\ndef score_filler_windows(model, waveform, speech):\n    """Yield the winning filler class and score for candidate windows."""\n    candidates = []\n    for offset in range(0, len(waveform), RATE // 4):\n        end = min(offset + RATE, len(waveform))\n        midpoint = (offset + end) / (2 * RATE)\n        if end - offset >= RATE // 2 and any(a <= midpoint <= b for a, b in speech):\n            candidates.append((offset, end))\n\n    model.eval()\n    with torch.inference_mode():\n        for i in range(0, len(candidates), 16):\n            batch = candidates[i:i + 16]\n            waveforms = np.stack([np.pad(waveform[a:b], (0, RATE - (b - a))) for a, b in batch])\n            scores, classes = model(torch.from_numpy(waveforms)).softmax(-1).max(-1)\n            for (a, b), score, label in zip(batch, scores.tolist(), classes.tolist()):\n                if label:\n                    yield {\'text\': LABELS[label], \'start_s\': a / RATE, \'end_s\': b / RATE,\n                           \'score\': score}\n\n\ndef transcript_filler_candidates(words, speech, duration):\n    """Generate explicit filler and speech-without-words candidates."""\n    words = sorted((max(0, a), min(duration, b), text.strip().lower().strip(\'.,!?;:\'))\n                   for text, a, b in words if 0 <= a < b <= duration + .1)\n    candidates = [(a, b, text) for a, b, text in words if text in LABELS[1:]]\n    for start, end in speech:\n        cursor = start\n        for a, b, _ in words:\n            if b <= start:\n                continue\n            if a >= end:\n                break\n            if .15 <= a - cursor <= 1.2:\n                candidates.append((cursor, a, None))\n            cursor = max(cursor, b)\n        if .15 <= end - cursor <= 1.2:\n            candidates.append((cursor, end, None))\n    return sorted(candidates)\n\n\ndef score_transcript_fillers(model, waveform, words, speech):\n    """Classify one-second windows centered on transcript and speech gaps."""\n    candidates = transcript_filler_candidates(words, speech, len(waveform) / RATE)\n    model.eval()\n    with torch.inference_mode():\n        for index in range(0, len(candidates), 16):\n            batch = candidates[index:index + 16]\n            clips = []\n            for a, b, _ in batch:\n                start = round((a + b - 1) * RATE / 2)\n                end = start + RATE\n                clips.append(np.pad(waveform[max(start, 0):end],\n                                    (max(0, -start), max(0, end - len(waveform)))))\n            scores, labels = model(torch.from_numpy(np.stack(clips))).softmax(-1).max(-1)\n            for (a, b, explicit), score, label in zip(batch, scores.tolist(), labels.tolist()):\n                if explicit or label:\n                    yield {\'text\': explicit or LABELS[label], \'start_s\': a, \'end_s\': b,\n                           \'score\': 1.0 if explicit else score}\n\n\ndef select_filler_events(windows, threshold):\n    """Suppress overlapping proposals in timestamp order."""\n    events = []\n    for event in windows:\n        if event[\'score\'] < threshold:\n            continue\n        if events and event[\'start_s\'] < events[-1][\'end_s\']:\n            if event[\'score\'] > events[-1][\'score\']:\n                events[-1] = event\n        else:\n            events.append(event)\n    return [{**event, \'score\': round(event[\'score\'], 4)} for event in events]\n\n\ndef detect_fillers(model, waveform, speech, threshold=0.85):\n    return select_filler_events(score_filler_windows(model, waveform, speech), threshold)\n', 'train_fillers.py': '"""Train a frozen wav2vec2 encoder\'s filler head on PodcastFillers clips."""\n\nimport argparse\nimport csv\nimport json\nimport os\nimport random\nimport re\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom torch.nn.parallel import DistributedDataParallel\nfrom torch.utils.data import DataLoader, Dataset, DistributedSampler\nfrom transformers import AutoModel\n\nfrom fillers import ENCODERS, LABELS, RATE, FillerClassifier, save_checkpoint, load_checkpoint, checkpoint_encoder\n\n\nclass PodcastClips(Dataset):\n    def __init__(self, root, annotations, split, limit=None):\n        paths = {}\n        for path in root.rglob(\'*.wav\'):\n            if path.name in paths:\n                raise ValueError(f\'duplicate clip filename: {path.name}\')\n            paths[path.name] = path\n        self.rows = []\n        with annotations.open(newline=\'\', encoding=\'utf-8-sig\') as stream:\n            for row in csv.DictReader(stream):\n                if row[\'clip_split_subset\'].lower() != split:\n                    continue\n                label = row[\'label_consolidated_vocab\'].strip().lower()\n                if label == \'none\':\n                    continue\n                self.rows.append((paths[row[\'clip_name\']], LABELS.index(label) if label in LABELS else 0))\n                if limit and len(self.rows) == limit:\n                    break\n        if not self.rows:\n            raise ValueError(f\'empty PodcastFillers split: {split}\')\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        path, label = self.rows[index]\n        waveform, rate = sf.read(path, dtype=\'float32\')\n        if rate != RATE or waveform.ndim != 1:\n            raise ValueError(f\'expected mono {RATE} Hz: {path}\')\n        return np.pad(waveform[:RATE], (0, max(0, RATE - len(waveform)))), label\n\n\ndef evaluate(model, clips, device, batch_size):\n    matrix = torch.zeros(3, 3, dtype=torch.int64)\n    model.eval()\n    with torch.inference_mode():\n        for audio, labels in DataLoader(clips, batch_size=batch_size):\n            predictions = model(audio.to(device)).argmax(1).cpu()\n            matrix += torch.bincount(3 * labels + predictions, minlength=9).reshape(3, 3)\n    return score_matrix(matrix)\n\n\ndef score_matrix(matrix):\n    scores = {}\n    for i, label in enumerate(LABELS):\n        tp = matrix[i, i].item()\n        precision = tp / max(1, matrix[:, i].sum().item())\n        recall = tp / max(1, matrix[i].sum().item())\n        scores[label] = {\'precision\': precision, \'recall\': recall,\n                         \'f1\': 2 * precision * recall / max(1e-10, precision + recall)}\n    return {\'confusion_matrix\': matrix.tolist(), \'labels\': scores}\n\n\ndef compare_transcript_baseline(model, clips, device, limit):\n    import whisper\n\n    transcription = whisper.load_model(\'small.en\', device=device, download_root=os.environ.get(\'SPEAKLY_MODEL_CACHE\'))\n    indices_by_label = [[i for i, (_, label) in enumerate(clips.rows) if label == index]\n                        for index in range(len(LABELS))]\n    rng = random.Random(42)\n    matrices = [torch.zeros(3, 3, dtype=torch.int64) for _ in range(2)]\n    with torch.inference_mode():\n        for indices in indices_by_label:\n            for i in rng.sample(indices, min(limit, len(indices))):\n                audio, label = clips[i]\n                transcript = transcription.transcribe(\n                    audio, language=\'en\', condition_on_previous_text=False,\n                    temperature=0, fp16=device.type == \'cuda\',\n                )[\'text\']\n                match = re.search(r\'\\b(um|uh)\\b\', transcript.lower())\n                baseline = LABELS.index(match.group()) if match else 0\n                learned = model(torch.from_numpy(audio[None, :]).to(device)).argmax(1).item()\n                matrices[0][label, baseline] += 1\n                matrices[1][label, learned] += 1\n    return {\'whisper_transcript\': score_matrix(matrices[0]),\n            \'wav2vec2_head\': score_matrix(matrices[1])}\n\n\ndef train(args):\n    rank = int(os.environ.get(\'RANK\', 0))\n    local_rank = int(os.environ.get(\'LOCAL_RANK\', 0))\n    world_size = int(os.environ.get(\'WORLD_SIZE\', 1))\n    device = torch.device(\'cpu\' if args.cpu else f\'cuda:{local_rank}\')\n    if not args.cpu:\n        torch.cuda.set_device(device)\n    if world_size > 1:\n        torch.distributed.init_process_group(\'gloo\' if args.cpu else \'nccl\')\n    try:\n        torch.manual_seed(42)\n        started = time.monotonic()\n        split_names = ((\'test\',) if args.evaluate_checkpoint else\n                       (\'train\', \'validation\') if args.validation_only else\n                       (\'train\', \'validation\', \'test\'))\n        splits = {split: PodcastClips(args.clips, args.csv, split, args.limit) for split in split_names}\n        encoder_name = checkpoint_encoder(args.evaluate_checkpoint) if args.evaluate_checkpoint else args.encoder\n        base_model, base_revision = ENCODERS[encoder_name]\n        model = FillerClassifier(AutoModel.from_pretrained(base_model, revision=base_revision),\n                                 unfreeze_last=args.unfreeze_last, encoder_name=encoder_name).to(device)\n        if args.evaluate_checkpoint:\n            if rank == 0:\n                load_checkpoint(model, args.evaluate_checkpoint)\n                result = {\'checkpoint\': str(args.evaluate_checkpoint), \'split_sizes\': {\'test\': len(splits[\'test\'])},\n                          \'test\': evaluate(model, splits[\'test\'], device, args.batch_size)}\n                if args.baseline_per_label:\n                    result[\'comparison\'] = compare_transcript_baseline(model, splits[\'test\'], device,\n                                                                       args.baseline_per_label)\n                args.output.mkdir(parents=True, exist_ok=True)\n                (args.output / \'metrics.json\').write_text(json.dumps(result, indent=2) + \'\\n\')\n                print(json.dumps(result), flush=True)\n            if world_size > 1:\n                torch.distributed.barrier()\n            return\n        wrapped = DistributedDataParallel(model, device_ids=None if args.cpu else [local_rank]) if world_size > 1 else model\n        sampler = DistributedSampler(splits[\'train\'], seed=42) if world_size > 1 else None\n        loader = DataLoader(splits[\'train\'], batch_size=args.batch_size, sampler=sampler,\n                            shuffle=sampler is None, num_workers=args.workers, pin_memory=not args.cpu)\n        parameters = [{\'params\': model.head.parameters(), \'lr\': 0.001}]\n        if args.unfreeze_last:\n            parameters.append({\'params\': [p for p in model.encoder.parameters() if p.requires_grad], \'lr\': 0.00001})\n        optimizer = torch.optim.AdamW(parameters)\n        best_score = -1\n        for epoch in range(args.epochs):\n            if sampler:\n                sampler.set_epoch(epoch)\n            wrapped.train()\n            for audio, targets in loader:\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=not args.cpu):\n                    logits = wrapped(audio.to(device, non_blocking=True))\n                    loss = torch.nn.functional.cross_entropy(\n                        logits.float(), targets.to(device, non_blocking=True)\n                    )\n                loss.backward()\n                optimizer.step()\n            if rank == 0:\n                validation = evaluate(model, splits[\'validation\'], device, args.batch_size)\n                score = sum(validation[\'labels\'][label][\'f1\'] for label in (\'um\', \'uh\')) / 2\n                print(json.dumps({\'epoch\': epoch + 1, \'validation\': validation}), flush=True)\n                if score > best_score:\n                    args.output.mkdir(parents=True, exist_ok=True)\n                    save_checkpoint(model, args.output / \'filler-head.pt\')\n                    best_score = score\n            if world_size > 1:\n                torch.distributed.barrier()\n        if rank == 0:\n            load_checkpoint(model, args.output / \'filler-head.pt\')\n            result = {\'base_model\': base_model, \'base_revision\': base_revision, \'world_size\': world_size,\n                      \'devices\': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n                      \'duration_s\': round(time.monotonic() - started, 2), \'unfreeze_last\': args.unfreeze_last,\n                      \'split_sizes\': {key: len(value) for key, value in splits.items()},\n                      \'best_validation_filler_f1\': best_score}\n            if not args.validation_only:\n                result[\'test\'] = evaluate(model, splits[\'test\'], device, args.batch_size)\n            if args.baseline_per_label and not args.validation_only:\n                try:\n                    result[\'comparison\'] = compare_transcript_baseline(\n                        model, splits[\'test\'], device, args.baseline_per_label\n                    )\n                except (ImportError, OSError, RuntimeError, ValueError) as exc:\n                    result[\'comparison_error\'] = str(exc)\n            (args.output / \'metrics.json\').write_text(json.dumps(result, indent=2) + \'\\n\')\n            print(json.dumps(result), flush=True)\n    finally:\n        if world_size > 1:\n            torch.distributed.destroy_process_group()\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--clips\', type=Path, required=True)\n    parser.add_argument(\'--csv\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--epochs\', type=int, default=3)\n    parser.add_argument(\'--batch-size\', type=int, default=24)\n    parser.add_argument(\'--workers\', type=int, default=2)\n    parser.add_argument(\'--limit\', type=int)\n    parser.add_argument(\'--baseline-per-label\', type=int, default=0)\n    parser.add_argument(\'--unfreeze-last\', type=int, default=0, help=\'Unfreeze the last N encoder layers\')\n    parser.add_argument(\'--encoder\', choices=tuple(ENCODERS), default=\'wav2vec2\')\n    parser.add_argument(\'--validation-only\', action=\'store_true\', help=\'Do not load or evaluate the test split\')\n    parser.add_argument(\'--evaluate-checkpoint\', type=Path, help=\'Evaluate one selected checkpoint on the test split\')\n    parser.add_argument(\'--cpu\', action=\'store_true\')\n    train(parser.parse_args())\n', 'kaggle_job.py': '"""Prepare the official research data and launch the private Kaggle run."""\n\nimport csv\nimport hashlib\nimport json\nimport subprocess\nimport sys\nimport tarfile\nfrom pathlib import Path\n\nRECORD = \'https://zenodo.org/records/7121457/files\'\nFILES = {\n    \'PodcastFillers.zip\': \'20e9f6b93b1b5c2e1c9d8b774ca28c33\',\n    \'PodcastFillers.csv\': \'68d8614ad562263c621b90dc81f06abd\',\n}\n\n\ndef download(root, name, checksum):\n    path = root / name\n    if not path.exists():\n        partial = path.with_suffix(path.suffix + \'.part\')\n        subprocess.run([\'curl\', \'-fL\', \'--retry\', \'4\', \'--retry-all-errors\', \'--connect-timeout\', \'30\',\n                        \'-C\', \'-\', \'-o\', str(partial), f\'{RECORD}/{name}\'], check=True)\n        partial.rename(path)\n    try:\n        verify_archive(path, checksum)\n    except ValueError:\n        path.unlink()\n        raise\n    return path\n\n\ndef verify_archive(path, checksum):\n    with path.open(\'rb\') as stream:\n        observed = hashlib.file_digest(stream, \'md5\').hexdigest()\n    if observed != checksum:\n        raise ValueError(f\'checksum mismatch: {path.name}\')\n\n\ndef skip_episode_volumes(root, size=6442450944, count=3):\n    # The pinned release stores every numbered clip entirely on the final volume.\n    # 7-Zip needs the other volumes\' sizes for offsets even when reading only clips.\n    for index in range(1, count + 1):\n        path = root / f\'PodcastFillers.z{index:02}\'\n        if not path.exists():\n            with path.open(\'xb\') as volume:\n                if index == 1:\n                    volume.write(b\'PK\\x07\\x08\')\n                volume.truncate(size)\n\n\ndef verify_clip_names(clips, csv_path):\n    with csv_path.open(newline=\'\', encoding=\'utf-8-sig\') as stream:\n        expected = {row[\'clip_name\'] for row in csv.DictReader(stream)}\n    observed = {path.name for path in clips.glob(\'*.wav\')}\n    if not expected or expected != observed:\n        raise ValueError(f\'clip inventory mismatch: {len(expected - observed)} missing, \'\n                         f\'{len(observed - expected)} unexpected\')\n\n\ndef extract_clips(root, sevenzip):\n    clips = root / \'clips\'\n    clips.mkdir(exist_ok=True)\n    subprocess.run([str(sevenzip), \'e\', str(root / \'PodcastFillers.zip\'), \'-r\',\n                    \'?????.wav\', f\'-o{clips}\', \'-aos\', \'-bsp0\'], check=True)\n    return clips\n\n\nEPISODE_VOLUME_CHECKSUM = \'399c44abb9ec35aa56d6fbf75043164b\'\n\n\ndef extract_episodes(root, sevenzip, csv_path):\n    members = {}\n    with csv_path.open(newline=\'\', encoding=\'utf-8-sig\') as stream:\n        for row in csv.DictReader(stream):\n            split = row[\'clip_split_subset\']\n            if split in (\'train\', \'validation\') and row[\'episode_split_subset\'] == split:\n                name = row[\'podcast_filename\']\n                members[name] = f\'PodcastFillers/audio/episode_mp3/{split}/{name}.mp3\'\n    include = root / \'episode-members.txt\'\n    include.write_text(\'\'.join(member + \'\\n\' for member in sorted(members.values())), encoding=\'utf-8\')\n    episodes = root / \'episodes\'\n    episodes.mkdir(exist_ok=True)\n    subprocess.run([str(sevenzip), \'e\', str(root / \'PodcastFillers.zip\'),\n                    f\'-i@{include}\', \'-scsUTF-8\', \'-spd\', f\'-o{episodes}\', \'-aoa\', \'-bsp0\'], check=True)\n    observed = {path.stem for path in episodes.glob(\'*.mp3\')}\n    if not members or observed != members.keys():\n        raise ValueError(\'original episode inventory differs from the official training and validation splits\')\n    for name in sorted(members):\n        audio = episodes / f\'{name}.mp3\'\n        subprocess.run([\'ffmpeg\', \'-v\', \'error\', \'-nostdin\', \'-y\', \'-i\', str(audio),\n                        \'-ac\', \'1\', \'-ar\', \'16000\', str(audio.with_suffix(\'.wav\'))], check=True)\n        audio.unlink()\n    return episodes\n\n\ndef install_extractor(root):\n    archive = root / \'7zip.tar.xz\'\n    subprocess.run([\'curl\', \'-fL\', \'--retry\', \'4\', \'-o\', str(archive),\n                    \'https://github.com/ip7z/7zip/releases/download/25.01/7z2501-linux-x64.tar.xz\'],\n                   check=True)\n    with archive.open(\'rb\') as stream:\n        digest = hashlib.file_digest(stream, \'sha256\').hexdigest()\n    if digest != \'4ca3b7c6f2f67866b92622818b58233dc70367be2f36b498eb0bdeaaa44b53f4\':\n        raise ValueError(\'7-Zip release checksum mismatch\')\n    with tarfile.open(archive) as release:\n        binary = root / \'7zz\'\n        binary.write_bytes(release.extractfile(\'7zz\').read())\n        binary.chmod(0o700)\n    return binary\n\n\ndef select_candidate(root):\n    candidates = (\'frozen\', \'partial\', \'wavlm\')\n    scores = {name: json.loads((root / name / \'metrics.json\').read_text())[\'best_validation_filler_f1\']\n              for name in candidates}\n    return max(candidates, key=lambda name: scores[name])\n\n\ndef run(root, frames_only=False, continuous=False):\n    import os\n\n    import torch\n\n    print(json.dumps({\'gpus\': [torch.cuda.get_device_name(i)\n                              for i in range(torch.cuda.device_count())]}), flush=True)\n    if torch.cuda.device_count() != 2 or any(\'T4\' not in torch.cuda.get_device_name(i)\n                                            for i in range(2)):\n        raise RuntimeError(\'Kaggle did not allocate two T4 GPUs\')\n    data = root / \'data\'\n    data.mkdir(exist_ok=True)\n    bundle_dir = os.getenv(\'SPEAKLY_CONTINUOUS_INPUT\') if continuous else None\n    bundle = Path(bundle_dir) if bundle_dir else None\n    required = {**FILES, \'PodcastFillers.z03\': EPISODE_VOLUME_CHECKSUM} if continuous else FILES\n    if bundle:\n        for name, checksum in required.items():\n            source_file = bundle / name\n            verify_archive(source_file, checksum)\n            staged = data / name\n            if staged.resolve() != source_file.resolve():\n                staged.unlink(missing_ok=True)\n                staged.symlink_to(source_file)\n        sevenzip = bundle / \'7zz\'\n        sevenzip_copy = root / \'7zz\'\n        binary = sevenzip.read_bytes()\n        if hashlib.sha256(binary).hexdigest() != \'a1860fdf0d6ec395e0e277e5222e9aa488747db4aa5c87d1ec879a0916ba0b2f\':\n            raise ValueError(\'7-Zip binary checksum mismatch\')\n        sevenzip_copy.write_bytes(binary)\n        sevenzip_copy.chmod(0o700)\n        sevenzip = sevenzip_copy\n    else:\n        for name, checksum in required.items():\n            download(data, name, checksum)\n        sevenzip = install_extractor(root)\n    skip_episode_volumes(data)\n    csv_path = data / \'PodcastFillers.csv\'\n    if continuous:\n        episodes = extract_episodes(data, sevenzip, csv_path)\n    else:\n        clips = extract_clips(data, sevenzip)\n        verify_clip_names(clips, csv_path)\n    if frames_only or continuous:\n        source = [\'--episodes\', str(episodes)] if continuous else [\'--clips\', str(clips)]\n        if bundle:\n            source += [\'--model-dir\', str(bundle / \'wavlm\')]\n        checkpoint = \'filler-continuous-head.pt\' if continuous else \'filler-frame-head.pt\'\n        subprocess.run([sys.executable, \'-m\', \'torch.distributed.run\', \'--standalone\',\n                        \'--nproc-per-node=2\', \'-m\', \'speech_analysis.train_filler_frames\',\n                        *source, \'--csv\', str(csv_path), \'--output\', f\'/kaggle/working/{checkpoint}\',\n                        \'--batch-size\', \'8\' if continuous else \'32\', \'--device\', \'cuda\'],\n                       check=True, env={**os.environ, \'PYTHONPATH\': str(root)})\n        return\n    output = Path(\'/kaggle/working/filler-comparison\')\n    for name, unfreeze, encoder in ((\'frozen\', 0, \'wav2vec2\'), (\'partial\', 1, \'wav2vec2\'),\n                                    (\'wavlm\', 0, \'wavlm\')):\n        subprocess.run([sys.executable, \'-m\', \'torch.distributed.run\', \'--standalone\',\n                        \'--nproc-per-node=2\', str(root / \'train_fillers.py\'), \'--clips\', str(clips),\n                        \'--csv\', str(data / \'PodcastFillers.csv\'), \'--output\', str(output / name),\n                        \'--unfreeze-last\', str(unfreeze), \'--encoder\', encoder, \'--validation-only\'], check=True)\n    selected = select_candidate(output)\n    (output / \'selected.json\').write_text(json.dumps({\'selected\': selected, \'criterion\': \'validation filler mean F1\'}))\n    subprocess.run([sys.executable, str(root / \'train_fillers.py\'), \'--clips\', str(clips),\n                    \'--csv\', str(data / \'PodcastFillers.csv\'), \'--output\', str(output / \'test\'),\n                    \'--evaluate-checkpoint\', str(output / selected / \'filler-head.pt\'),\n                    \'--baseline-per-label\', \'32\'], check=True)\n    sys.path.insert(0, str(root))\n    from speech_analysis.kaggle_disfluency import run as run_disfluency\n    run_disfluency(root)\n\n\ndef notebook(owner, destination, frames_only=False, continuous=False):\n    source = Path(__file__).parent\n    files = {name: (source / name).read_text() for name in\n             (\'fillers.py\', \'train_fillers.py\', \'kaggle_job.py\')}\n    files.update({f\'speech_analysis/{name}\': (source / name).read_text() for name in\n                  (\'fillers.py\', \'disfluency.py\', \'disfluency_model.py\', \'train_disfluency.py\',\n                   \'evaluate_disfluency.py\', \'kaggle_disfluency.py\', \'train_filler_frames.py\')})\n    command = "run-continuous" if continuous else "run-frames" if frames_only else "run"\n    dependencies = ["transformers==4.57.6", "soundfile"]\n    if not (continuous or frames_only):\n        dependencies += ["openai-whisper==20250625", "pyarrow==22.0.0", "librosa==1.0.1"]\n    network_check = (\n        \'import importlib.metadata, socket, time\\n\'\n        \'def wait_for_dns(host):\\n\'\n        \'    for delay in (0, 15, 30, 60):\\n\'\n        \'        time.sleep(delay)\\n\'\n        \'        try:\\n\'\n        \'            socket.getaddrinfo(host, 443)\\n\'\n        \'            return\\n\'\n        \'        except socket.gaierror as error:\\n\'\n        \'            print(f"Kaggle DNS lookup failed for {host}: {error}", flush=True)\\n\'\n        \'    raise RuntimeError(f"Kaggle worker cannot resolve {host} after retries. Check Internet is On, then retry on a fresh worker.")\\n\'\n        \'def missing(requirement):\\n\'\n        \'    package, _, version = requirement.partition("==")\\n\'\n        \'    try:\\n\'\n        \'        installed = importlib.metadata.version(package)\\n\'\n        \'    except importlib.metadata.PackageNotFoundError:\\n\'\n        \'        return True\\n\'\n        \'    return bool(version and installed != version)\\n\'\n        \'import os\\n\'\n        f\'bundles = list(Path("/kaggle/input").glob("**/speakly-continuous-input.json")) if {continuous!r} else []\\n\'\n        \'if len(bundles) > 1:\\n\'\n        \'    raise RuntimeError("Attach only one Speakly continuous input bundle.")\\n\'\n        \'if bundles:\\n\'\n        \'    bundle = bundles[0].parent\\n\'\n        \'    os.environ["SPEAKLY_CONTINUOUS_INPUT"] = str(bundle)\\n\'\n        \'for host in (() if bundles else ("zenodo.org", "huggingface.co")):\\n\'\n        \'    wait_for_dns(host)\\n\'\n        f\'needed = [requirement for requirement in {dependencies!r} if missing(requirement)]\\n\'\n        \'if needed:\\n\'\n        \'    if bundles:\\n\'\n        \'        pip_source = ["--no-index", "--find-links", str(bundle / "wheels")]\\n\'\n        \'    else:\\n\'\n        \'        wait_for_dns("pypi.org")\\n\'\n        \'        pip_source = []\\n\'\n        \'    subprocess.run([sys.executable, "-m", "pip", "install", *pip_source, *needed], check=True)\\n\'\n    )\n    code = (\n        \'import subprocess, sys\\nfrom pathlib import Path\\n\'\n        \'root = Path("/kaggle/temp/speakly")\\nroot.mkdir(parents=True, exist_ok=True)\\n\'\n        f\'files = {files!r}\\n\'\n        \'for name, content in files.items():\\n    path = root / name\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    path.write_text(content)\\n\'\n        + network_check\n        + f\'subprocess.run([sys.executable, str(root / "kaggle_job.py"), {command!r}], check=True)\\n\' \n    )\n    destination.mkdir(parents=True, exist_ok=True)\n    (destination / \'train.ipynb\').write_text(json.dumps({\n        \'cells\': [{\'cell_type\': \'code\', \'source\': code, \'metadata\': {}, \'outputs\': [],\n                   \'execution_count\': None}],\n        \'metadata\': {\'kernelspec\': {\'display_name\': \'Python 3\', \'language\': \'python\', \'name\': \'python3\'}},\n        \'nbformat\': 4, \'nbformat_minor\': 4,\n    }))\n    slug = \'speakly-filler-frames\' if frames_only else \'speakly-filler-research\'\n    title = \'Speakly Filler Frames\' if frames_only else \'Speakly Filler Research\'\n    if continuous:\n        slug, title = \'speakly-filler-continuous\', \'Speakly Filler Continuous\'\n    (destination / \'kernel-metadata.json\').write_text(json.dumps({\n        \'id\': f\'{owner}/{slug}\', \'title\': title,\n        \'code_file\': \'train.ipynb\', \'language\': \'python\', \'kernel_type\': \'notebook\',\n        \'is_private\': True, \'enable_gpu\': True, \'enable_internet\': True,\n        \'dataset_sources\': [], \'competition_sources\': [], \'kernel_sources\': [],\n    }))\n\n\nif __name__ == \'__main__\':\n    if sys.argv[1:] in ([\'run\'], [\'run-frames\'], [\'run-continuous\']):\n        run(Path(__file__).parent, frames_only=sys.argv[1] == \'run-frames\', continuous=sys.argv[1] == \'run-continuous\')\n    elif len(sys.argv) == 4 and sys.argv[1] == \'--frames\':\n        notebook(sys.argv[2], Path(sys.argv[3]), frames_only=True)\n    elif len(sys.argv) == 4 and sys.argv[1] == \'--continuous\':\n        notebook(sys.argv[2], Path(sys.argv[3]), continuous=True)\n    elif len(sys.argv) == 3:\n        notebook(sys.argv[1], Path(sys.argv[2]))\n    else:\n        raise SystemExit(\'Usage: python kaggle_job.py KAGGLE_OWNER OUTPUT_DIRECTORY\')\n', 'speech_analysis/fillers.py': '"""Optional research classifier and fixed-window inference."""\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nLABELS = (\'other\', \'um\', \'uh\')\nBASE_MODEL = \'facebook/wav2vec2-base\'\nBASE_REVISION = \'0b5b8e868dd84f03fd87d01f9c4ff0f080fecfe8\'\nENCODERS = {\'wav2vec2\': (BASE_MODEL, BASE_REVISION),\n            \'wavlm\': (\'microsoft/wavlm-base-plus\', \'4c66d4806a428f2e922ccfa1a962776e232d487b\')}\nRATE = 16000\n\n\nclass FillerClassifier(nn.Module):\n    def __init__(self, encoder, unfreeze_last=0, encoder_name=\'wav2vec2\'):\n        super().__init__()\n        if encoder_name not in ENCODERS:\n            raise ValueError(\'unsupported encoder\')\n        self.encoder_name = encoder_name\n        self.encoder = encoder.eval().requires_grad_(False)\n        layers = encoder.encoder.layers\n        if not 0 <= unfreeze_last <= len(layers):\n            raise ValueError(\'invalid number of encoder layers to unfreeze\')\n        self.unfreeze_last = unfreeze_last\n        if unfreeze_last:\n            for layer in layers[-unfreeze_last:]:\n                layer.requires_grad_(True)\n        self.head = nn.Linear(encoder.config.hidden_size, len(LABELS))\n\n    def forward(self, waveform):\n        normalized = (waveform - waveform.mean(dim=-1, keepdim=True)) / (\n            waveform.var(dim=-1, keepdim=True, unbiased=False) + 1e-7\n        ).sqrt()\n        if not self.unfreeze_last:\n            self.encoder.eval()\n        with torch.set_grad_enabled(self.training and bool(self.unfreeze_last)):\n            hidden = self.encoder(normalized).last_hidden_state.mean(dim=1)\n        return self.head(hidden)\n\n\ndef save_checkpoint(model, path):\n    state = {\'head\': model.head.state_dict(), \'unfreeze_last\': model.unfreeze_last,\n             \'encoder_name\': model.encoder_name}\n    if model.unfreeze_last:\n        state[\'encoder\'] = model.encoder.state_dict()\n    torch.save(state, path)\n\n\ndef load_checkpoint(model, path):\n    state = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if state.get(\'kind\') == \'frame_filler\' and not getattr(model, \'frame_output\', False):\n        raise ValueError(\'frame checkpoint requires the frame classifier\')\n    if \'head\' not in state:\n        model.head.load_state_dict(state)\n        return\n    if state.get(\'encoder_name\', \'wav2vec2\') != model.encoder_name:\n        raise ValueError(\'checkpoint encoder does not match model\')\n    model.head.load_state_dict(state[\'head\'])\n    if \'encoder\' in state:\n        model.encoder.load_state_dict(state[\'encoder\'])\n\n\ndef checkpoint_encoder(path):\n    state = torch.load(path, map_location=\'cpu\', weights_only=True)\n    name = state.get(\'encoder_name\', \'wav2vec2\')\n    if name not in ENCODERS:\n        raise ValueError(\'unsupported checkpoint encoder\')\n    return name\n\n\ndef score_filler_windows(model, waveform, speech):\n    """Yield the winning filler class and score for candidate windows."""\n    candidates = []\n    for offset in range(0, len(waveform), RATE // 4):\n        end = min(offset + RATE, len(waveform))\n        midpoint = (offset + end) / (2 * RATE)\n        if end - offset >= RATE // 2 and any(a <= midpoint <= b for a, b in speech):\n            candidates.append((offset, end))\n\n    model.eval()\n    with torch.inference_mode():\n        for i in range(0, len(candidates), 16):\n            batch = candidates[i:i + 16]\n            waveforms = np.stack([np.pad(waveform[a:b], (0, RATE - (b - a))) for a, b in batch])\n            scores, classes = model(torch.from_numpy(waveforms)).softmax(-1).max(-1)\n            for (a, b), score, label in zip(batch, scores.tolist(), classes.tolist()):\n                if label:\n                    yield {\'text\': LABELS[label], \'start_s\': a / RATE, \'end_s\': b / RATE,\n                           \'score\': score}\n\n\ndef transcript_filler_candidates(words, speech, duration):\n    """Generate explicit filler and speech-without-words candidates."""\n    words = sorted((max(0, a), min(duration, b), text.strip().lower().strip(\'.,!?;:\'))\n                   for text, a, b in words if 0 <= a < b <= duration + .1)\n    candidates = [(a, b, text) for a, b, text in words if text in LABELS[1:]]\n    for start, end in speech:\n        cursor = start\n        for a, b, _ in words:\n            if b <= start:\n                continue\n            if a >= end:\n                break\n            if .15 <= a - cursor <= 1.2:\n                candidates.append((cursor, a, None))\n            cursor = max(cursor, b)\n        if .15 <= end - cursor <= 1.2:\n            candidates.append((cursor, end, None))\n    return sorted(candidates)\n\n\ndef score_transcript_fillers(model, waveform, words, speech):\n    """Classify one-second windows centered on transcript and speech gaps."""\n    candidates = transcript_filler_candidates(words, speech, len(waveform) / RATE)\n    model.eval()\n    with torch.inference_mode():\n        for index in range(0, len(candidates), 16):\n            batch = candidates[index:index + 16]\n            clips = []\n            for a, b, _ in batch:\n                start = round((a + b - 1) * RATE / 2)\n                end = start + RATE\n                clips.append(np.pad(waveform[max(start, 0):end],\n                                    (max(0, -start), max(0, end - len(waveform)))))\n            scores, labels = model(torch.from_numpy(np.stack(clips))).softmax(-1).max(-1)\n            for (a, b, explicit), score, label in zip(batch, scores.tolist(), labels.tolist()):\n                if explicit or label:\n                    yield {\'text\': explicit or LABELS[label], \'start_s\': a, \'end_s\': b,\n                           \'score\': 1.0 if explicit else score}\n\n\ndef select_filler_events(windows, threshold):\n    """Suppress overlapping proposals in timestamp order."""\n    events = []\n    for event in windows:\n        if event[\'score\'] < threshold:\n            continue\n        if events and event[\'start_s\'] < events[-1][\'end_s\']:\n            if event[\'score\'] > events[-1][\'score\']:\n                events[-1] = event\n        else:\n            events.append(event)\n    return [{**event, \'score\': round(event[\'score\'], 4)} for event in events]\n\n\ndef detect_fillers(model, waveform, speech, threshold=0.85):\n    return select_filler_events(score_filler_windows(model, waveform, speech), threshold)\n', 'speech_analysis/disfluency.py': '"""Conservative parser for annotated reparandum/repair pairs in DisfluencySpeech."""\n\nimport re\nimport io\n\nPAIR = re.compile(r\'\\[([^\\[\\]+]+)\\+([^\\[\\]]+)\\]\')\nWORDS = re.compile(r"[a-z]+(?:\'[a-z]+)?", re.IGNORECASE)\n\n\ndef labeled_repairs(transcript):\n    """Parse bracketed pairs; unmarked repairs are not negative examples."""\n    events = []\n    for match in PAIR.finditer(transcript):\n        before = match.group(1).strip(\' ,\')\n        after = match.group(2).strip(\' ,\')\n        left = WORDS.findall(before.lower())\n        right = WORDS.findall(after.lower())\n        if not left or not right:\n            continue\n        repetition = len(right) >= len(left) and left == right[:len(left)] and not before.endswith(\'-\')\n        events.append({\'type\': \'repetition\' if repetition else \'repair\',\n                       \'reparandum\': before, \'repair\': after})\n    return events\n\n\ndef clip_labels(transcript):\n    """Predict presence of parsed annotation types, not clinical disfluency."""\n    kinds = {event[\'type\'] for event in labeled_repairs(transcript)}\n    return (int(\'repetition\' in kinds), int(\'repair\' in kinds))\n\n\nclass DisfluencyClips:\n    """Load short annotated full utterances; do not truncate clip-level labels."""\n\n    def __init__(self, paths, limit=None, max_seconds=12):\n        import pyarrow.parquet as pq\n        import soundfile as sf\n\n        self.rows = []\n        for path in paths:\n            parquet = pq.ParquetFile(path)\n            for batch in parquet.iter_batches(batch_size=32, columns=[\'audio\', \'transcript_annotated\']):\n                for row in batch.to_pylist():\n                    payload = row[\'audio\'][\'bytes\']\n                    info = sf.info(io.BytesIO(payload))\n                    if info.channels != 1 or not 0.5 <= info.duration <= max_seconds:\n                        continue\n                    self.rows.append((payload, clip_labels(row[\'transcript_annotated\'])))\n                    if limit and len(self.rows) >= limit:\n                        return\n        if not self.rows:\n            raise ValueError(\'no suitable disfluency utterances\')\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        import librosa\n        import numpy as np\n        import soundfile as sf\n        import torch\n\n        payload, labels = self.rows[index]\n        waveform, rate = sf.read(io.BytesIO(payload), dtype=\'float32\')\n        if rate != 16000:\n            waveform = librosa.resample(waveform, orig_sr=rate, target_sr=16000)\n        return torch.from_numpy(np.ascontiguousarray(waveform)), torch.tensor(labels, dtype=torch.float32)\n', 'speech_analysis/disfluency_model.py': '"""Research-only clip classifier for two parsed annotation types."""\n\nimport torch\nfrom torch import nn\nfrom torch.nn.utils.rnn import pad_sequence\n\n\ndef collate(rows):\n    waves, labels = zip(*rows)\n    return pad_sequence(waves, batch_first=True), torch.tensor([len(x) for x in waves]), torch.stack(labels)\n\n\ndef annotation_scores(logits, targets):\n    prediction = logits >= 0\n    reference = targets.bool()\n    result = {}\n    for i, name in enumerate((\'repetition\', \'repair\')):\n        tp = int((prediction[:, i] & reference[:, i]).sum())\n        fp = int((prediction[:, i] & ~reference[:, i]).sum())\n        fn = int((~prediction[:, i] & reference[:, i]).sum())\n        precision = tp / (tp + fp) if tp + fp else 0.0\n        recall = tp / (tp + fn) if tp + fn else 0.0\n        result[name] = {\'tp\': tp, \'fp\': fp, \'fn\': fn, \'precision\': precision, \'recall\': recall,\n                        \'f1\': 2 * precision * recall / (precision + recall) if precision + recall else 0.0}\n    return result\n\n\ndef predict_segments(model, waveform, sample_rate=16000, window_seconds=12):\n    """Return uncalibrated clip scores, not event detections or weaknesses."""\n    output = []\n    model.eval()\n    size = sample_rate * window_seconds\n    with torch.inference_mode():\n        for start in range(0, len(waveform), size):\n            segment = waveform[start:start + size]\n            if len(segment) < sample_rate // 2:\n                continue\n            logits = model(segment.unsqueeze(0), torch.tensor([len(segment)], device=segment.device))\n            probs = logits.sigmoid()[0].tolist()\n            output.append({\'start_s\': start / sample_rate, \'end_s\': (start + len(segment)) / sample_rate,\n                           \'probabilities\': {\'repetition\': probs[0], \'repair\': probs[1]}})\n    return output\n\n\nclass DisfluencyClassifier(nn.Module):\n    def __init__(self, encoder):\n        super().__init__()\n        self.encoder = encoder.eval().requires_grad_(False)\n        self.head = nn.Linear(encoder.config.hidden_size, 2)\n\n    def forward(self, waveform, lengths):\n        positions = torch.arange(waveform.shape[1], device=waveform.device)[None, :]\n        mask = positions < lengths[:, None]\n        values = waveform * mask\n        mean = values.sum(-1, keepdim=True) / lengths[:, None]\n        centered = (waveform - mean) * mask\n        variance = centered.square().sum(-1, keepdim=True) / lengths[:, None]\n        normalized = centered / (variance + 1e-7).sqrt()\n        self.encoder.eval()\n        with torch.no_grad():\n            hidden = self.encoder(normalized, attention_mask=mask.long()).last_hidden_state\n            frame_lengths = self.encoder._get_feat_extract_output_lengths(lengths).clamp(min=1, max=hidden.shape[1])\n            frame_mask = torch.arange(hidden.shape[1], device=waveform.device)[None, :] < frame_lengths[:, None]\n            pooled = (hidden * frame_mask.unsqueeze(-1)).sum(dim=1) / frame_lengths[:, None]\n        return self.head(pooled)\n', 'speech_analysis/train_disfluency.py': '"""Research training for presence of parsed disfluency annotations per clip."""\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\n\nimport torch\nfrom torch.nn.parallel import DistributedDataParallel\nfrom torch.utils.data import DataLoader\nfrom torch.utils.data.distributed import DistributedSampler\nfrom transformers import AutoModel\n\nfrom .disfluency import DisfluencyClips\nfrom .disfluency_model import DisfluencyClassifier, annotation_scores, collate\nfrom .fillers import ENCODERS\n\n\ndef evaluate(model, data, device, batch_size):\n    predictions, targets = [], []\n    model.eval()\n    with torch.inference_mode():\n        for waves, lengths, labels in DataLoader(data, batch_size=batch_size, collate_fn=collate):\n            predictions.append(model(waves.to(device), lengths.to(device)).cpu())\n            targets.append(labels)\n    return annotation_scores(torch.cat(predictions), torch.cat(targets))\n\n\ndef train(args):\n    if {p.resolve() for p in args.train} & {p.resolve() for p in args.validation}:\n        raise ValueError(\'train and validation paths must be distinct\')\n    rank = int(os.environ.get(\'RANK\', \'0\'))\n    local_rank = int(os.environ.get(\'LOCAL_RANK\', \'0\'))\n    world_size = int(os.environ.get(\'WORLD_SIZE\', \'1\'))\n    device = torch.device(\'cpu\' if args.cpu else f\'cuda:{local_rank}\')\n    if not args.cpu:\n        torch.cuda.set_device(device)\n    if world_size > 1:\n        torch.distributed.init_process_group(\'gloo\' if args.cpu else \'nccl\')\n    try:\n        torch.manual_seed(42)\n        train_data = DisfluencyClips(args.train, limit=args.limit)\n        validation = DisfluencyClips(args.validation, limit=args.limit)\n        model_name, revision = ENCODERS[args.encoder]\n        model = DisfluencyClassifier(AutoModel.from_pretrained(model_name, revision=revision)).to(device)\n        wrapped = DistributedDataParallel(model, device_ids=None if args.cpu else [local_rank]) if world_size > 1 else model\n        sampler = DistributedSampler(train_data, seed=42) if world_size > 1 else None\n        loader = DataLoader(train_data, batch_size=args.batch_size, sampler=sampler,\n                            shuffle=sampler is None, collate_fn=collate)\n        optimizer = torch.optim.AdamW(model.head.parameters(), lr=0.001)\n        best = -1.0\n        for epoch in range(args.epochs):\n            if sampler:\n                sampler.set_epoch(epoch)\n            wrapped.train()\n            for waves, lengths, labels in loader:\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=not args.cpu):\n                    logits = wrapped(waves.to(device), lengths.to(device))\n                    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), labels.to(device))\n                loss.backward()\n                optimizer.step()\n            if rank == 0:\n                scores = evaluate(model, validation, device, args.batch_size)\n                score = (scores[\'repetition\'][\'f1\'] + scores[\'repair\'][\'f1\']) / 2\n                print(json.dumps({\'epoch\': epoch + 1, \'validation\': scores}), flush=True)\n                if score > best:\n                    best = score\n                    args.output.mkdir(parents=True, exist_ok=True)\n                    torch.save({\'head\': model.head.state_dict(), \'encoder\': args.encoder, \'revision\': revision,\n                                \'label_semantics\': \'presence of automatically parsed bracket annotation\'},\n                               args.output / \'disfluency-head.pt\')\n                    (args.output / \'metrics.json\').write_text(json.dumps({\n                        \'validation\': scores, \'best_validation_mean_f1\': best,\n                        \'train_clips\': len(train_data), \'validation_clips\': len(validation),\n                        \'encoder\': args.encoder, \'revision\': revision, \'world_size\': world_size,\n                        \'limitations\': \'single acted speaker; labels are parsed annotation presence\'}, indent=2) + \'\\n\')\n            if world_size > 1:\n                torch.distributed.barrier()\n    finally:\n        if world_size > 1:\n            torch.distributed.destroy_process_group()\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--train\', type=Path, nargs=\'+\', required=True)\n    parser.add_argument(\'--validation\', type=Path, nargs=\'+\', required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--encoder\', choices=tuple(ENCODERS), default=\'wav2vec2\')\n    parser.add_argument(\'--epochs\', type=int, default=3)\n    parser.add_argument(\'--batch-size\', type=int, default=2)\n    parser.add_argument(\'--limit\', type=int)\n    parser.add_argument(\'--cpu\', action=\'store_true\')\n    train(parser.parse_args())\n', 'speech_analysis/evaluate_disfluency.py': '"""Evaluate a selected research checkpoint on annotation-presence labels."""\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport torch\nfrom transformers import AutoModel\n\nfrom .disfluency import DisfluencyClips\nfrom .disfluency_model import DisfluencyClassifier\nfrom .fillers import ENCODERS\nfrom .train_disfluency import evaluate\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--test\', nargs=\'+\', type=Path, required=True)\n    parser.add_argument(\'--checkpoint\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--batch-size\', type=int, default=2)\n    parser.add_argument(\'--limit\', type=int)\n    parser.add_argument(\'--cpu\', action=\'store_true\')\n    args = parser.parse_args()\n    device = torch.device(\'cpu\' if args.cpu else \'cuda:0\')\n    checkpoint = torch.load(args.checkpoint, map_location=\'cpu\', weights_only=True)\n    base, revision = ENCODERS[checkpoint[\'encoder\']]\n    if checkpoint[\'revision\'] != revision:\n        raise ValueError(\'checkpoint revision mismatch\')\n    model = DisfluencyClassifier(AutoModel.from_pretrained(base, revision=revision)).to(device)\n    model.head.load_state_dict(checkpoint[\'head\'])\n    dataset = DisfluencyClips(args.test, limit=args.limit)\n    result = {\'samples\': len(dataset), \'test\': evaluate(model, dataset, device, args.batch_size),\n              \'limitations\': \'single acted speaker; targets are automatically parsed annotations\'}\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(result, indent=2) + \'\\n\')\n    print(json.dumps(result))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'speech_analysis/kaggle_disfluency.py': '"""Stage pinned DisfluencySpeech data and run research training on two GPUs."""\n\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nREVISION = \'b7da294fe3a70dd96df6640893f2a5dfc2c87638\'\nFILES = {\n    \'train-00000-of-00003.parquet\': \'6285da585435e76adefe43c3142852fd92a6be0c8c820486d941421a8638b4ae\',\n    \'train-00001-of-00003.parquet\': \'bfb907061b59e14fe9f9c2f6e9a00e13fe1cf531e50e6987c7cd34aecad81887\',\n    \'train-00002-of-00003.parquet\': \'16bc5a61a9ff54923b78ca86d17c7028e5f36d9b3655637f3ffd64e73cf37426\',\n    \'validation-00000-of-00001.parquet\': \'693ba2a06301d9cbafea52eae0664f8c776467e9b559ad9e25b6756f236f6a9c\',\n    \'test-00000-of-00001.parquet\': \'e78837be393206fc0ae6a0cefb4ec8c942c3e26401640eb8af67cf0dac21b4b0\',\n}\n\n\ndef stage(path, name, expected):\n    target = path / name\n    if not target.exists():\n        temporary = target.with_suffix(\'.part\')\n        url = f\'https://huggingface.co/datasets/amaai-lab/DisfluencySpeech/resolve/{REVISION}/data/{name}\'\n        subprocess.run([\'curl\', \'-fL\', \'--retry\', \'4\', \'--connect-timeout\', \'30\', \'-o\', str(temporary), url],\n                       check=True)\n        temporary.rename(target)\n    with target.open(\'rb\') as stream:\n        if hashlib.file_digest(stream, \'sha256\').hexdigest() != expected:\n            raise ValueError(f\'dataset checksum mismatch: {name}\')\n    return target\n\n\ndef run(root):\n    import torch\n\n    devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]\n    if len(devices) != 2 or any(\'T4\' not in name for name in devices):\n        raise RuntimeError(\'expected two T4 GPUs\')\n    data = root / \'disfluency-data\'\n    data.mkdir(exist_ok=True)\n    paths = {name: stage(data, name, checksum) for name, checksum in FILES.items()}\n    output = Path(\'/kaggle/working/disfluency-research\')\n    env = dict(os.environ, PYTHONPATH=str(root), OMP_NUM_THREADS=\'2\')\n    train = [str(path) for name, path in paths.items() if name.startswith(\'train-\')]\n    validation = next(path for name, path in paths.items() if name.startswith(\'validation-\'))\n    test = next(path for name, path in paths.items() if name.startswith(\'test-\'))\n    subprocess.run([sys.executable, \'-m\', \'torch.distributed.run\', \'--standalone\', \'--nproc-per-node=2\',\n                    \'-m\', \'speech_analysis.train_disfluency\', \'--train\', *train,\n                    \'--validation\', str(validation), \'--output\', str(output), \'--batch-size\', \'2\'],\n                   check=True, env=env)\n    subprocess.run([sys.executable, \'-m\', \'speech_analysis.evaluate_disfluency\', \'--test\', str(test),\n                    \'--checkpoint\', str(output / \'disfluency-head.pt\'), \'--output\', str(output / \'test.json\')],\n                   check=True, env=env)\n    (output / \'run.json\').write_text(json.dumps({\'devices\': devices, \'dataset_revision\': REVISION}, indent=2))\n', 'speech_analysis/train_filler_frames.py': '"""Train a research frame head using the official PodcastFillers timestamp labels."""\n\nimport argparse\nimport csv\nimport json\nimport os\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom torch.utils.data import DataLoader, Dataset, DistributedSampler\nfrom transformers import AutoModel\n\nfrom .fillers import ENCODERS, LABELS, RATE, FillerClassifier\n\n\ndef frame_targets(start, end, label, frame_count):\n    # WavLM\'s convolutional extractor has a 400-sample receptive field\n    # and produces one frame per 320 samples.\n    centers = (torch.arange(frame_count) * 320 + 199.5) / RATE\n    return ((centers >= start) & (centers < end)).long() * label\n\n\ndef continuous_targets(intervals, frame_count):\n    centers = (torch.arange(frame_count) * 320 + 199.5) / RATE\n    observed = torch.zeros((3, frame_count), dtype=torch.bool)\n    for start, end, label in intervals:\n        observed[label] |= (centers >= start) & (centers < end)\n    labels = observed.long().argmax(dim=0)\n    labels[observed.sum(dim=0) != 1] = -100\n    return labels\n\n\nclass ContinuousWindows(Dataset):\n    """Read four-second windows from predecoded mono 16 kHz episode WAVs."""\n\n    samples = 4 * RATE\n    frame_count = (samples - 400) // 320 + 1\n\n    def __init__(self, episodes, csv_file, split, limit=None):\n        self.intervals = defaultdict(list)\n        with csv_file.open(newline=\'\', encoding=\'utf-8-sig\') as source:\n            for row in csv.DictReader(source):\n                if row[\'clip_split_subset\'].lower() != split or row[\'episode_split_subset\'].lower() != split:\n                    continue\n                path = episodes / (row[\'podcast_filename\'] + \'.wav\')\n                label = row[\'label_consolidated_vocab\'].lower()\n                if label == \'none\':\n                    continue\n                start = float(row[\'event_start_inepisode\'])\n                end = float(row[\'event_end_inepisode\'])\n                if 0 <= start < end:\n                    self.intervals[path].append((start, end, LABELS.index(label) if label in LABELS else 0))\n        self.windows = []\n        for path, events in self.intervals.items():\n            if not path.is_file():\n                raise FileNotFoundError(path)\n            for start, end, _ in events:\n                offset = max(0, round(((start + end) / 2 - 2) * RATE))\n                self.windows.append((path, offset))\n        self.windows = list(dict.fromkeys(self.windows))[:limit]\n        if not self.windows:\n            raise ValueError(f\'empty {split} split\')\n\n    def __len__(self):\n        return len(self.windows)\n\n    def __getitem__(self, index):\n        path, offset = self.windows[index]\n        with sf.SoundFile(path) as episode:\n            if episode.samplerate != RATE or episode.channels != 1:\n                raise ValueError(\'expected 16 kHz mono episodes\')\n            if offset >= len(episode):\n                raise ValueError(f\'annotation outside episode: {path}\')\n            episode.seek(offset)\n            audio = episode.read(self.samples, dtype=\'float32\')\n        start = offset / RATE\n        end = start + len(audio) / RATE\n        intervals = [(left - start, min(right, end) - start, label)\n                     for left, right, label in self.intervals[path] if right > start and left < end]\n        labels = continuous_targets(intervals, self.frame_count)\n        return torch.from_numpy(np.pad(audio, (0, self.samples - len(audio)))), labels\n\n\nclass AnnotatedClips(Dataset):\n    def __init__(self, clips, csv_file, split, limit=None):\n        with csv_file.open(newline=\'\', encoding=\'utf-8-sig\') as source:\n            self.rows = [row for row in csv.DictReader(source)\n                         if row[\'clip_split_subset\'].lower() == split\n                         and row[\'label_consolidated_vocab\'].lower() != \'none\']\n        self.rows = self.rows[:limit]\n        if not self.rows:\n            raise ValueError(f\'empty {split} split\')\n        self.clips = clips\n        self.training = split == \'train\'\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        audio, rate = sf.read(self.clips / row[\'clip_name\'], dtype=\'float32\')\n        if rate != RATE or audio.ndim != 1:\n            raise ValueError(\'expected 16 kHz mono clips\')\n        audio = np.pad(audio[:RATE], (0, max(0, RATE - len(audio))))\n        label = row[\'label_consolidated_vocab\'].lower()\n        label = LABELS.index(label) if label in LABELS else 0\n        start, end = (float(row[k]) for k in (\'event_start_inclip\', \'event_end_inclip\'))\n        shift = int(torch.randint(-1920, 1921, ()).item()) if self.training else 0\n        if shift > 0:\n            audio = np.pad(audio[:-shift], (shift, 0))\n        elif shift < 0:\n            audio = np.pad(audio[-shift:], (0, -shift))\n        return torch.from_numpy(audio), start + shift / RATE, end + shift / RATE, label\n\n\nclass FrameClassifier(FillerClassifier):\n    frame_output = True\n\n    def forward(self, waveform):\n        mean = waveform.mean(dim=-1, keepdim=True)\n        scale = (waveform.var(dim=-1, keepdim=True, unbiased=False) + 1e-7).sqrt()\n        self.encoder.eval()\n        with torch.no_grad():\n            frames = self.encoder((waveform - mean) / scale).last_hidden_state\n        return self.head(frames)\n\n\ndef train(args):\n    torch.manual_seed(42)\n    torch.set_num_threads(4)\n    world = int(os.getenv(\'WORLD_SIZE\', \'1\'))\n    rank = int(os.getenv(\'RANK\', \'0\'))\n    local_rank = int(os.getenv(\'LOCAL_RANK\', \'0\'))\n    device = torch.device(f\'cuda:{local_rank}\' if args.device == \'cuda\' else args.device)\n    if device.type == \'cuda\':\n        torch.cuda.set_device(device)\n    if world > 1:\n        torch.distributed.init_process_group(\'nccl\' if device.type == \'cuda\' else \'gloo\')\n    name, revision = ENCODERS[\'wavlm\']\n    encoder = (AutoModel.from_pretrained(args.model_dir, local_files_only=True) if args.model_dir\n               else AutoModel.from_pretrained(name, revision=revision))\n    model = FrameClassifier(encoder, encoder_name=\'wavlm\').to(device)\n    optimizer = torch.optim.AdamW(model.head.parameters(), lr=.001)\n    dataset = ContinuousWindows if args.episodes else AnnotatedClips\n    datasets = {split: dataset(args.episodes or args.clips, args.csv, split, args.limit)\n                for split in (\'train\', \'validation\')}\n    sampler = DistributedSampler(datasets[\'train\']) if world > 1 else None\n    loaders = {split: DataLoader(data, shuffle=split == \'train\' and sampler is None,\n                                 sampler=sampler if split == \'train\' else None,\n                                 batch_size=args.batch_size)\n               for split, data in datasets.items()}\n    wrapped = (torch.nn.parallel.DistributedDataParallel(\n        model, device_ids=[local_rank] if device.type == \'cuda\' else None)\n        if world > 1 else model)\n    best = -1\n    for epoch in range(args.epochs):\n        if sampler is not None:\n            sampler.set_epoch(epoch)\n        for split, loader in loaders.items():\n            if split == \'validation\' and rank != 0:\n                continue\n            training = split == \'train\'\n            model.train(training)\n            matrix = torch.zeros((3, 3), dtype=torch.long)\n            for audio, *annotations in loader:\n                with torch.set_grad_enabled(training):\n                    scores = (wrapped if training else model)(audio.to(device))\n                    if args.episodes:\n                        targets = annotations[0].to(device)\n                    else:\n                        targets = torch.stack([frame_targets(a, b, label, scores.shape[1])\n                                               for a, b, label in zip(*annotations)]).to(device)\n                    if training:\n                        weights = scores.new_tensor([1., 3., 3.])\n                        loss = torch.nn.functional.cross_entropy(\n                            scores.flatten(0, 1), targets.flatten(), weight=weights, reduction=\'sum\')\n                        known = targets != -100\n                        loss = loss / weights[targets[known]].sum().clamp_min(1)\n                        optimizer.zero_grad(set_to_none=True)\n                        loss.backward()\n                        optimizer.step()\n                known = targets != -100\n                bins = 3 * targets[known] + scores.detach().argmax(-1)[known]\n                matrix += torch.bincount(bins, minlength=9).reshape(3, 3).cpu()\n            f1 = [2 * int(matrix[i, i]) / max(1, int(matrix[i].sum() + matrix[:, i].sum()))\n                  for i in (1, 2)]\n            if training and world > 1:\n                device_matrix = matrix.to(device)\n                torch.distributed.all_reduce(device_matrix)\n                matrix = device_matrix.cpu()\n                f1 = [2 * int(matrix[i, i]) / max(1, int(matrix[i].sum() + matrix[:, i].sum()))\n                      for i in (1, 2)]\n            if rank != 0:\n                continue\n            print(json.dumps({\'epoch\': epoch + 1, \'split\': split, \'frames\': matrix.tolist(),\n                              \'filler_mean_f1\': sum(f1) / 2}), flush=True)\n            if not training and sum(f1) / 2 > best:\n                best = sum(f1) / 2\n                args.output.parent.mkdir(parents=True, exist_ok=True)\n                torch.save({\'head\': model.head.state_dict(), \'encoder_name\': \'wavlm\', \'revision\': revision,\n                            \'kind\': \'frame_filler\', \'window_samples\': ContinuousWindows.samples if args.episodes else RATE,\n                            \'validation_frame_f1\': best}, args.output)\n        if world > 1:\n            torch.distributed.barrier()\n    if world > 1:\n        torch.distributed.destroy_process_group()\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    source = parser.add_mutually_exclusive_group(required=True)\n    source.add_argument(\'--clips\', type=Path)\n    source.add_argument(\'--episodes\', type=Path)\n    parser.add_argument(\'--csv\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--epochs\', type=int, default=3)\n    parser.add_argument(\'--batch-size\', type=int, default=16)\n    parser.add_argument(\'--limit\', type=int)\n    parser.add_argument(\'--device\', default=\'cpu\')\n    parser.add_argument(\'--model-dir\', type=Path)\n    train(parser.parse_args())\n'}
for name, content in files.items():
    path = root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
import importlib.metadata, socket, time
def wait_for_dns(host):
    for delay in (0, 15, 30, 60):
        time.sleep(delay)
        try:
            socket.getaddrinfo(host, 443)
            return
        except socket.gaierror as error:
            print(f"Kaggle DNS lookup failed for {host}: {error}", flush=True)
    raise RuntimeError(f"Kaggle worker cannot resolve {host} after retries. Check Internet is On, then retry on a fresh worker.")
def missing(requirement):
    package, _, version = requirement.partition("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return True
    return bool(version and installed != version)
import os
bundles = list(Path("/kaggle/input").glob("**/speakly-continuous-input.json")) if True else []
if len(bundles) > 1:
    raise RuntimeError("Attach only one Speakly continuous input bundle.")
if bundles:
    bundle = bundles[0].parent
    os.environ["SPEAKLY_CONTINUOUS_INPUT"] = str(bundle)
for host in (() if bundles else ("zenodo.org", "huggingface.co")):
    wait_for_dns(host)
needed = [requirement for requirement in ['transformers==4.57.6', 'soundfile'] if missing(requirement)]
if needed:
    if bundles:
        pip_source = ["--no-index", "--find-links", str(bundle / "wheels")]
    else:
        wait_for_dns("pypi.org")
        pip_source = []
    subprocess.run([sys.executable, "-m", "pip", "install", *pip_source, *needed], check=True)
subprocess.run([sys.executable, str(root / "kaggle_job.py"), 'run-continuous'], check=True)
